# 📦 Pallet Carton Detection — YOLOv8 Training on Google Colab

Train custom YOLOv8 model for **Angled / Overhead Pallet Carton Counting**.

> **Note for Colab**: Make sure GPU is enabled:
> `Runtime` -> `Change runtime type` -> select **T4 GPU**.

In [ ]:
# Step 1: Check GPU & Install Dependencies
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 📥 Step 2: Load Dataset
Choose **Option A (Roboflow Direct Export)** OR **Option B (Upload Zip file)**.

In [ ]:
# OPTION A: Direct Roboflow Download (Paste your code from Roboflow Export)
# -------------------------------------------------------------------------
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# version = project.version(1)
# dataset = version.download("yolov8")
# data_yaml_path = f"{dataset.location}/data.yaml"

In [ ]:
# OPTION B: Upload Dataset Zip Manually
import os, glob, zipfile
from google.colab import files

print("Upload your dataset zip file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/dataset')
        print(f"Extracted {filename} to /content/dataset")

# Auto-locate data.yaml
found_yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True) + glob.glob('/content/**/data.yaml', recursive=True)
if found_yamls:
    data_yaml_path = found_yamls[0]
    print(f"Found data.yaml at: {data_yaml_path}")
else:
    # Create default data.yaml if not present
    data_yaml_path = '/content/dataset/data.yaml'
    with open(data_yaml_path, 'w') as f:
        f.write("""
train: /content/dataset/train/images
val: /content/dataset/valid/images
test: /content/dataset/test/images
nc: 1
names: ['top_carton']
""".strip())
    print(f"Created default {data_yaml_path}")

## 🚀 Step 3: Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load pre-trained weights (yolov8n for speed or yolov8s for higher accuracy)
model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml_path,
    epochs=120,
    imgsz=640,
    batch=16,
    name='pallet_carton_v8n',
    patience=25,
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    translate=0.10,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)

## 📊 Step 4: Validate Model & Review Metrics

In [ ]:
# Run validation
metrics = model.val()

print("="*40)
print(f"mAP @ 0.50:       {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
print(f"mAP @ 0.50-0.95:  {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
print(f"Precision:        {metrics.box.p[0]:.4f}")
print(f"Recall:           {metrics.box.r[0]:.4f}")
print("="*40)

In [ ]:
# Display training result plots
from IPython.display import Image, display
import glob

for plot_path in glob.glob('runs/detect/pallet_carton_v8n*/results.png') + glob.glob('runs/detect/pallet_carton_v8n*/confusion_matrix.png'):
    print(f"Displaying: {plot_path}")
    display(Image(plot_path))

## 💾 Step 5: Download `best.pt` Weights File
Download the trained weights to use in your local project.

In [ ]:
import glob
from google.colab import files

# Find latest best.pt
best_weights = glob.glob('runs/detect/pallet_carton_v8n*/weights/best.pt')
if best_weights:
    latest_best = best_weights[-1]
    print(f"Downloading {latest_best}...")
    files.download(latest_best)
else:
    print("Error: best.pt weights file not found.")